# Door Background Removal with Nova Canvas

Uses Amazon Nova Canvas (Bedrock) to remove the background from door photos.

**Cost:** ~$0.08 per image

**No GPU needed** — just an API call.

In [ ]:
!pip install -q boto3 pillow matplotlib

In [ ]:
import os
import json
import base64
import boto3
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Nova Canvas is available in us-east-1
bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")
print("Ready.")

In [ ]:
IMAGE_DIR = "doors"
os.makedirs("output", exist_ok=True)
images = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
print(f"Found {len(images)} images: {images}")

In [ ]:
def remove_background(image_path):
    """Remove background using Nova Canvas."""
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode()

    response = bedrock.invoke_model(
        modelId="amazon.nova-canvas-v1:0",
        body=json.dumps({
            "taskType": "BACKGROUND_REMOVAL",
            "backgroundRemovalParams": {
                "image": {
                    "source": {
                        "bytes": img_b64
                    }
                }
            }
        })
    )

    result = json.loads(response["body"].read())
    output_b64 = result["images"][0]
    img_bytes = base64.b64decode(output_b64)
    
    # Nova Canvas returns PNG with alpha
    from io import BytesIO
    return Image.open(BytesIO(img_bytes))

In [ ]:
# Test on first image
if images:
    test_path = os.path.join(IMAGE_DIR, images[0])
    print(f"Processing: {images[0]}")
    result = remove_background(test_path)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(Image.open(test_path))
    axes[0].set_title("Original")
    axes[0].axis("off")
    axes[1].imshow(result)
    axes[1].set_title("Background removed")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Process all images
for img_name in images:
    img_path = os.path.join(IMAGE_DIR, img_name)
    print(f"Processing: {img_name}")
    
    result = remove_background(img_path)
    
    out_name = os.path.splitext(img_name)[0] + "_cutout.png"
    result.save(os.path.join("output", out_name))
    print(f"  Saved: output/{out_name}")
    
    # Show
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(Image.open(img_path))
    axes[0].set_title("Original")
    axes[0].axis("off")
    axes[1].imshow(result)
    axes[1].set_title("Cutout")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

## Next Steps

1. Pick the best front and back cutouts
2. Perspective-correct to a rectangle (homography)
3. Generate a flat-plane GLB with front/back textures
4. Display in webapp with model-viewer (user toggles front/back)